# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the [FAIR² mass spectrometry dataset](https://doi.org/10.71728/senscience.vq4a-28xa/) using the `mlcroissant` library. All record sets and fields are referenced by their `@id` per the dataset's Croissant schema.

### Dataset Source
The dataset source is defined via a Croissant schema JSON-LD file at the following URL:

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
We load dataset metadata with `mlcroissant.Dataset`, display the dataset name and description, and prepare to examine the available record sets and fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the URL to the Croissant metadata
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset and metadata
dataset = mlc.Dataset(croissant_url)

# Print basic dataset information (without subscripting dataset.metadata)
print(f"Dataset title: {dataset.metadata.name}")
print(f"Dataset description: {dataset.metadata.description}")

## 2. Data Overview
Let's examine all available record sets and their fields, using their `@id` attributes. This is essential for referencing them precisely in subsequent data extraction and processing steps.

In [ ]:
# List all record sets by their `@id` and show their available fields with IDs
if not dataset.metadata.record_set:
    print("No record sets found in metadata. If this occurs, the Croissant schema may place record sets on distribution objects or as part of the distribution.")
else:
    for rs in dataset.metadata.record_set:
        print(f"Record Set: {rs['@id']}")
        if 'field' in rs:
            print("  Fields:")
            for fld in rs['field']:
                print(f"    - {fld['@id']} (name: {fld.get('name', '')})")
        else:
            print("  No fields listed.")

In [ ]:
# If the dataset places its record sets under the distribution key, enumerate them that way
record_set_ids = []
record_set_field_map = {}
if not dataset.metadata.record_set and hasattr(dataset.metadata, 'distribution'):
    for dist in dataset.metadata.distribution:
        if hasattr(dist, 'record_set') and dist.record_set:
            for rs in dist.record_set:
                record_set_ids.append(rs['@id'])
                fields = [fld['@id'] for fld in rs.get('field', [])]
                record_set_field_map[rs['@id']] = fields
                print(f"Distribution Record Set: {rs['@id']}")
                print(f"  Fields: {fields}")
else:
    # Standard top-level record set
    if dataset.metadata.record_set:
        for rs in dataset.metadata.record_set:
            record_set_ids.append(rs['@id'])
            fields = [fld['@id'] for fld in rs.get('field', [])]
            record_set_field_map[rs['@id']] = fields
            print(f"Record Set: {rs['@id']}")
            print(f"  Fields: {fields}")

# For further cells, define a record set id to use
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Using main record set: {main_record_set_id}")
else:
    main_record_set_id = None
    print("No record sets found. Please check your dataset.")

## 3. Data Extraction
Now we'll load data from one or more record sets into pandas DataFrames. All record sets and their `@id` values are used for referencing.

> **Note:** If the record set contains many rows and columns, consider loading only a subset or specific fields for efficiency.

In [ ]:
# Extract all available record set data as DataFrames
dfs = {}
for rs_id in record_set_ids:
    try:
        print(f"Loading records from record set {rs_id} ...")
        # 'dataset.records' yields dicts per row
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dfs[rs_id] = df
        print(f"  Loaded {len(df)} rows and columns: {list(df.columns)}")
    except Exception as e:
        print(f"  Failed to load {rs_id}: {e}")

# We'll use the main_record_set_id for EDA. Show the first few rows.
if main_record_set_id is not None and main_record_set_id in dfs:
    display_cols = list(dfs[main_record_set_id].columns)
    print(f"Columns in record set '{main_record_set_id}': {display_cols}")
    dfs[main_record_set_id].head()
else:
    print("No loaded DataFrame to display.")

## 4. Exploratory Data Analysis (EDA)
Let's process the data: we'll select a numeric field by `@id`, filter, normalize, and group if a categorical field is present.

> For demonstration, field and group IDs should be chosen from those shown in previous outputs. Update `numeric_field_id` and `group_field_id` as appropriate for your schema.

In [ ]:
# Pick a numeric field and group field by @id
# Replace these string literals with actual @ids from your schema.

numeric_field_id = None
group_field_id = None

# You may need to examine the columns first
if main_record_set_id and main_record_set_id in dfs:
    df = dfs[main_record_set_id]
    print("Available columns:", list(df.columns))
    # Simple heuristic: try to pick a float/integer field if present
    for col in df.columns:
        if df[col].dtype in [int, float, 'int64', 'float64'] or pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    for col in df.columns:
        # Try to find a likely categorical/group field
        if col.lower().startswith('group') or col.lower().startswith('sample') or pd.api.types.is_object_dtype(df[col]):
            # Avoid choosing the same as numeric field
            if col != numeric_field_id:
                group_field_id = col
                break

    if numeric_field_id:
        print(f"Using numeric field: {numeric_field_id}")
    else:
        print("No numeric field detected. Please update `numeric_field_id` manually.")

    # Proceed with analysis if numeric field found
    if numeric_field_id:
        # Filter records > threshold
        threshold = 10
        filtered = df[df[numeric_field_id] > threshold]
        print(f"Filtered records in '{main_record_set_id}' where {numeric_field_id} > {threshold}: {len(filtered)} rows")
        display_cols = [numeric_field_id]
        print(filtered[display_cols].head())

        # Normalize
        col_norm = f"{numeric_field_id}_normalized"
        filtered[col_norm] = (filtered[numeric_field_id] - filtered[numeric_field_id].mean()) / filtered[numeric_field_id].std()
        print(f"Normalized {numeric_field_id}:")
        print(filtered[[numeric_field_id, col_norm]].head())

        # Group by
        if group_field_id and group_field_id in filtered.columns:
            grouped = filtered.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped.head())
        else:
            print("No suitable group field detected for grouping.")
    else:
        print("No suitable numeric field found.")
else:
    print("No usable data for EDA.")

## 5. Visualization
Visualize data distributions or relations between fields. Here, we plot a histogram and (if possible) boxplot of the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dfs and numeric_field_id:
    df = dfs[main_record_set_id]
    # Histogram
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30)
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group if present
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable data for visualization.")

## 6. Conclusion
- We have loaded the FAIR² mass spectrometry dataset using the `mlcroissant` library by referencing record sets and fields with their `@id`.
- We explored metadata, record set fields, and loaded tabular data for analysis.
- Basic EDA and visualizations give an overview of data distributions and group-wise behavior.

You can now tailor advanced analytics or machine learning steps using this workflow as a foundation.